# Speculative Decoding

> Autoregressive generation has a physical limit: each token must wait for the previous one to finish computing. KV Cache reduces the computation per step, but cannot change the fundamentally serial nature of the process.
>
> Speculative Decoding bypasses this limit with an elegant idea: let a small model guess a sequence of tokens first, then have the large model verify them all in parallel. Correct guesses are kept; incorrect ones trigger a rollback. On straightforward text, each step can advance 2-3 tokens, effectively doubling or tripling the speed while producing output identical to the original model.

Speculative Decoding builds on a statistical fact: the probability distribution for most tokens in text is highly concentrated. For example, after "The capital of France is", the next token is almost certainly "Paris". Even a small model with one-tenth the parameters of the large model can guess correctly.

This means a small model can first generate a batch of candidate tokens, and the large model can verify them all in a single forward pass. Tokens that pass verification are kept; those that fail trigger a rollback from the rejection point. The key constraint is that the output must be exactly the same as using the large model alone -- speculation must not sacrifice quality.

## 1. The Serial Bottleneck of Autoregressive Generation

Recall autoregressive generation:

```
Step 1: Input [BOS]           -> predict token_1
Step 2: Input [BOS, tok_1]    -> predict token_2
Step 3: Input [BOS, tok_1, tok_2] -> predict token_3
```

Each step depends on the result of the previous step -- a serial dependency that cannot be parallelized. Even with KV Cache, each step must still wait for the previous one to complete.

But if we could guess the next token, we could start computing subsequent positions early. This is the intuition behind speculative decoding:

```
Normal:  compute tok_1 -> wait -> compute tok_2 -> wait -> compute tok_3
Speculative: guess tok_1=5 -> simultaneously compute tok_2(assuming 1=5) and tok_3(assuming 1=5, 2=3)
             ^ if the guess is correct, we get 3 tokens at once
```

## 2. The Complete Speculative Decoding Pipeline

Two models are needed:
- **Draft Model**: small and fast. For example, 0.5B parameters.
- **Target Model**: large and slow. For example, 7B or 70B parameters.

The process has three steps:

```
Step 1: Draft guesses K tokens
  Input: [The, weather]
  Draft generates autoregressively: [is, nice, today, !]  (K=4)

Step 2: Target verifies in one forward pass
  Input: [The, weather, is, nice, today, !]
  Target outputs probabilities for each position
  Check: position 2 guessed "is"   -> Target agrees  YES
         position 3 guessed "nice" -> Target agrees  YES
         position 4 guessed "today"-> Target prefers "great"  NO

Step 3: Accept / Reject
  Keep: is, nice (first 2)
  Discard: today, ! (from position 3 onward)
  Sample from Target: great

  Result: 1 Target forward pass produces 3 tokens (is, nice, great)
```

## 3. Acceptance and Rejection Criteria

Acceptance is not simply checking whether the draft's top prediction matches the target's. Instead, it uses a probability ratio:

$$
\text{accept\_prob}(x) = \min\left(1, \frac{p_{\text{target}}(x)}{p_{\text{draft}}(x)}\right)
$$

- If the Target is more confident about this token than the Draft -> always accept
- If the Draft was overconfident but the Target disagrees -> may reject
- This rule guarantees that the final output distribution is exactly the same as using the Target model alone

In [ ]:
# Simulate acceptance/rejection logic
import torch

def simulate_accept_reject(draft_probs, target_probs, draft_tokens, seed=42):
    torch.manual_seed(seed)
    accepted = []
    for dp, tp, tok in zip(draft_probs, target_probs, draft_tokens):
        accept_prob = min(1.0, tp / dp) if dp > 0 else 0.0
        if torch.rand(1).item() < accept_prob:
            accepted.append(tok)
        else:
            break
    return accepted

draft_tokens = [15, 23, 8, 42]
draft_probs = [0.8, 0.7, 0.6, 0.5]
target_probs = [0.9, 0.8, 0.2, 0.1]

print('=== Simulating Acceptance Decisions in Speculative Decoding ===')
print(f'Draft tokens:     {draft_tokens}')
print(f'Draft probs:      {draft_probs}')
print(f'Target probs:     {target_probs}')
print()

for i in range(len(draft_tokens)):
    ratio = target_probs[i] / draft_probs[i]
    ap = min(1.0, ratio)
    status = 'Always accept' if ratio >= 1 else f'{ap:.0%} accept'
    print(f'  token {draft_tokens[i]}: p_t/p_d = {ratio:.2f} -> {status}')

accepted = simulate_accept_reject(draft_probs, target_probs, draft_tokens)
print(f'\nAccepted this run: {accepted} (first {len(accepted)} tokens)')

## 4. Multiple Experiments: Actual Acceptance Rate

A single experiment has randomness. Below we run 1000 experiments, tallying the actual acceptance rate at each position to verify it matches the theoretical value.

In [ ]:
# Run 1000 experiments, tally acceptance rate per position
import torch

n_trials = 1000
position_accepted = [0] * len(draft_tokens)

draft_tokens_sim = [15, 23, 8, 42]
draft_probs_sim = torch.tensor([0.8, 0.7, 0.6, 0.5])
target_probs_sim = torch.tensor([0.9, 0.8, 0.2, 0.1])

for trial in range(n_trials):
    result = simulate_accept_reject(draft_probs_sim, target_probs_sim, draft_tokens_sim, seed=trial)
    for pos in range(len(result)):
        position_accepted[pos] += 1

print(f'=== Statistics from {n_trials} Experiments ===')
print(f'{"Pos":>4}  {"Draft prob":>10}  {"Target prob":>11}  {"Theory rate":>11}  {"Actual rate":>11}')
print('-' * 55)
for i in range(len(draft_tokens_sim)):
    dp = draft_probs_sim[i].item()
    tp = target_probs_sim[i].item()
    theory = min(1.0, tp / dp) if dp > 0 else 0
    actual = position_accepted[i] / n_trials
    print(f'{i:>4}  {dp:>10.2f}  {tp:>11.2f}  {theory:>11.2%}  {actual:>11.2%}')

print(f'\nTheoretical and actual values are close, confirming the acceptance rule is correct.')

## 5. Implementing Draft and Target Models

To fully demonstrate speculative decoding in code, we need two models: a small Draft model and a larger Target model. Using the MiniGPT architecture from earlier, we instantiate two versions with different sizes. Both models are trained on the same data -- the Draft is smaller and faster, the Target is larger and more accurate.

In [ ]:
import torch
import torch.nn as nn

import math

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=32, num_heads=2, num_layers=2, max_seq_len=64):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        pe = torch.zeros(max_seq_len, d_model)
        pos = torch.arange(max_seq_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe)
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=d_model, nhead=num_heads,
                dim_feedforward=4*d_model, batch_first=True, activation='relu')
            for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        batch, seq = x.shape
        h = self.token_emb(x) + self.pe[:seq, :]
        mask = nn.Transformer.generate_square_subsequent_mask(seq, device=x.device)
        for block in self.blocks:
            h = block(h, src_mask=mask, is_causal=True)
        return self.lm_head(self.ln_f(h))

VOCAB = 20  # vocabulary size
# Draft: small model (d=32, 2 layers)
draft_model = MiniGPT(VOCAB, d_model=32, num_heads=2, num_layers=2)
# Target: large model (d=64, 4 layers)
target_model = MiniGPT(VOCAB, d_model=64, num_heads=4, num_layers=4)

print(f'Draft parameters: {sum(p.numel() for p in draft_model.parameters()):,}')
print(f'Target parameters: {sum(p.numel() for p in target_model.parameters()):,}')
print(f'Target is ~{sum(p.numel() for p in target_model.parameters()) / sum(p.numel() for p in draft_model.parameters()):.1f}x larger than Draft')

In [ ]:
# Train both models

import torch
import torch.nn.functional as F

import torch.nn as nn

def make_data(n=500, seq_len=12, vocab=VOCAB, pattern_len=8):
    data = []
    for i in range(n):
        seq = [(i + j) % pattern_len + 1 for j in range(seq_len)]
        data.append(seq)
    return torch.tensor(data)

train_data = make_data()

def train_model(model, data, epochs=30, lr=0.01, name=''):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for ep in range(epochs):
        total_loss = 0
        for i in range(0, len(data), 32):
            batch = data[i:i+32]
            logits = model(batch[:, :-1])
            loss = F.cross_entropy(logits.reshape(-1, VOCAB), batch[:, 1:].reshape(-1))
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item()
        if (ep+1) % 10 == 0:
            print(f'  {name} Epoch {ep+1}: loss={total_loss:.4f}')
    return model

print('Training Draft model:')
draft_model = train_model(draft_model, train_data, epochs=30, lr=0.01, name='Draft')
print('\nTraining Target model:')
target_model = train_model(target_model, train_data, epochs=30, lr=0.005, name='Target')

## 6. Complete Speculative Decoding Implementation

Now with both models ready, we implement the full speculative decoding pipeline: Draft guesses K tokens -> Target verifies in one forward pass -> accept/reject based on the probability ratio.

In [ ]:
import torch
import torch.nn.functional as F

import torch.nn as nn

def speculative_decode(draft, target, prompt, K=4, max_tokens=30, temperature=1.0):
    """
    Speculative decoding: Draft guesses K tokens, Target verifies in one pass.
    Returns: (generated token list, target call count, total tokens, accepted count)
    """
    generated = prompt.clone()
    target_calls = 0
    total_draft_tokens = 0
    total_accepted = 0

    for _ in range(max_tokens // max(K, 1) + 1):
        if generated.shape[1] >= max_tokens + prompt.shape[1]:
            break

        # Step 1: Draft autoregressively guesses K tokens
        draft_tokens = []
        draft_probs_list = []
        draft_input = generated.clone()

        draft.eval()
        with torch.no_grad():
            for _ in range(K):
                logits = draft(draft_input)
                probs = F.softmax(logits[0, -1, :] / max(temperature, 0.01), dim=-1)
                token = torch.multinomial(probs, 1).unsqueeze(0)
                draft_tokens.append(token.item())
                draft_probs_list.append(probs[token.item()].item())
                draft_input = torch.cat([draft_input, token], dim=1)

        total_draft_tokens += K

        # Step 2: Target verifies in one forward pass
        # Construct input: original sequence + K draft tokens
        verify_input = torch.cat([generated, torch.tensor([draft_tokens])], dim=1)
        target_calls += 1

        target.eval()
        with torch.no_grad():
            target_logits = target(verify_input)

        # Step 3: Check accept/reject for each position
        accepted_count = 0
        for pos in range(K):
            t_pos = generated.shape[1] - 1 + pos  # corresponding position in target logits
            target_probs = F.softmax(target_logits[0, t_pos, :] / max(temperature, 0.01), dim=-1)
            dp = draft_probs_list[pos]
            tp = target_probs[draft_tokens[pos]].item()

            accept_prob = min(1.0, tp / dp) if dp > 0 else 0

            if torch.rand(1).item() < accept_prob:
                accepted_count += 1
            else:
                # Rejected: sample a token from the target distribution
                new_token = torch.multinomial(target_probs, 1).unsqueeze(0)
                generated = torch.cat([generated,
                    torch.tensor([draft_tokens[:accepted_count]], dtype=torch.long),
                    new_token], dim=1)
                total_accepted += accepted_count
                break
        else:
            # All accepted
            generated = torch.cat([generated,
                torch.tensor([draft_tokens], dtype=torch.long)], dim=1)
            total_accepted += K

        if generated.shape[1] >= max_tokens + prompt.shape[1]:
            break

    return generated, target_calls, total_accepted, total_draft_tokens

print('Speculative decoding function defined.')

In [ ]:
# Compare speculative decoding vs. normal decoding
import torch
import torch.nn.functional as F

import torch.nn as nn

torch.manual_seed(42)
prompt = torch.tensor([[2, 3]])
max_new = 20

# Normal decoding (using Target model)
def normal_decode(model, prompt, max_tokens=20, temperature=1.0):
    generated = prompt.clone()
    calls = 0
    model.eval()
    with torch.no_grad():
        for _ in range(max_tokens):
            logits = model(generated)
            probs = F.softmax(logits[0, -1, :] / max(temperature, 0.01), dim=-1)
            token = torch.multinomial(probs, 1).unsqueeze(0)
            generated = torch.cat([generated, token], dim=1)
            calls += 1
    return generated, calls

torch.manual_seed(42)
normal_result, normal_calls = normal_decode(target_model, prompt, max_tokens=max_new)

# Speculative decoding (K=4)
torch.manual_seed(42)
spec_result, spec_calls, spec_accepted, spec_draft = speculative_decode(
    draft_model, target_model, prompt, K=4, max_tokens=max_new)

print(f'=== Comparison (generating ~{max_new} tokens) ===')
print(f'Normal decoding:')
print(f'  Target calls: {normal_calls}')
print(f'  Result: {normal_result[0].tolist()}')
print(f'Speculative decoding (K=4):')
print(f'  Target calls: {spec_calls}')
print(f'  Draft tokens guessed: {spec_draft}')
print(f'  Accepted: {spec_accepted}')
print(f'  Acceptance rate: {spec_accepted/spec_draft*100:.1f}%')
print(f'  Result: {spec_result[0].tolist()}')
print(f'\nTarget call reduction: {normal_calls} -> {spec_calls} ({(1-spec_calls/normal_calls)*100:.0f}%)')

## 7. Speedup Analysis

The speedup of speculative decoding depends on the Draft model's accuracy. Below we experiment with different values of K and observe how the acceptance rate and speedup change.

In [ ]:
# Experiment with different K values
import torch

torch.manual_seed(42)
prompt = torch.tensor([[1, 2, 3]])

print(f'{"K":>4}  {"Target calls":>12}  {"Draft total":>10}  {"Accepted":>8}  {"Accept rate":>11}  {"Speedup":>8}')
print('-' * 60)

for K in [2, 3, 4, 5, 6, 8]:
    torch.manual_seed(42)
    result, calls, accepted, total_draft = speculative_decode(
        draft_model, target_model, prompt, K=K, max_tokens=20)
    n_generated = result.shape[1] - prompt.shape[1]
    accept_rate = accepted / total_draft * 100 if total_draft > 0 else 0
    # Speedup = tokens generated / target calls
    speedup = n_generated / calls if calls > 0 else 0
    print(f'{K:>4}  {calls:>12}  {total_draft:>10}  {accepted:>8}  {accept_rate:>10.1f}%  {speedup:>7.2f}x')

print('\nLarger K means more tokens guessed per round, but the acceptance rate may drop.')
print('The optimal K depends on the Draft model accuracy.')

## 8. Speedup Visualization

Below we simulate the theoretical speedup under different acceptance rates and compare with data from the actual models.

In [ ]:
# Theoretical speedup: different acceptance rates and K combinations
import numpy as np
import matplotlib.pyplot as plt

accept_rates = np.linspace(0.1, 0.99, 50)

plt.figure(figsize=(8, 5))
for K in [2, 4, 6, 8]:
    # Theory: expected tokens per round follows geometric distribution
    # Expected tokens = sum(alpha^k, k=0..K-1) + 1 (correction token)
    # Simplified: speedup ~ expected_accepted / 1
    expected = [(1 - a**K) / (1 - a) for a in accept_rates]
    plt.plot(accept_rates * 100, expected, label=f'K={K}', linewidth=2)

plt.xlabel('Draft Acceptance Rate (%)')
plt.ylabel('Tokens per Target Forward Pass')
plt.title('Theoretical Speedup of Speculative Decoding')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print('At 90% acceptance rate + K=4, each Target forward pass yields ~3.5 tokens on average.')

## 9. Variants of Speculative Decoding

Several variants exist in practice. The core idea is the same; only the source of draft tokens differs:

| Variant | Draft Source | Characteristics |
|---------|-------------|----------------|
| Standard Speculative Decoding | Independent small model | Requires training/deploying a separate small model |
| Self-Speculative | Model's own early layers | No extra model needed |
| Medusa | Multiple prediction heads | Attach extra linear heads to the model |
| Eagle | Small feature transformation network | Use target features to predict future tokens |
| Lookahead | n-gram matching | Find patterns in already-generated text |

Medusa is particularly elegant: it attaches several extra linear heads to the last layer of the LLM, each predicting the token at position t+k. No separate draft model is needed -- only a few small heads to train. One forward pass simultaneously predicts the next 4 tokens, which are then verified and accepted.

In [ ]:
# Medusa concept illustration
print('=== Medusa Head Architecture ===')
print()
print('Standard LLM:')
print('  hidden -> lm_head -> predict token_{t+1}')
print()
print('Medusa LLM:')
print('  hidden -> lm_head       -> predict token_{t+1}')
print('  hidden -> medusa_head_0 -> predict token_{t+2}')
print('  hidden -> medusa_head_1 -> predict token_{t+3}')
print('  hidden -> medusa_head_2 -> predict token_{t+4}')
print()
print('One forward pass predicts the next 4 tokens simultaneously.')
print('Advantage: no extra model needed, just a few small linear layers.')
print('Disadvantage: Medusa heads need separate training; accuracy is lower than an independent draft model.')

## 10. When to Use Speculative Decoding

Speculative decoding is not a universal solution:

**Good fit**: Generating "formulaic" content (code, translation, summarization) -> Draft guesses accurately -> significant speedup.
**Poor fit**: Creative writing -> Draft guesses poorly -> frequent rejections -> may actually be slower.

Key insight: Speculative decoding does not reduce total computation. It converts serial computation into "small model serial + large model parallel verification." The large model's computation stays the same, but waiting time is reduced.

In [ ]:
# Simulate speculative decoding performance in different scenarios
import torch

print('=== Speculative Decoding Performance in Different Scenarios ===')
print()

# Scenario 1: Simple pattern (high acceptance rate)
print('Scenario 1: Simple repeating pattern (high acceptance rate)')
torch.manual_seed(42)
prompt1 = torch.tensor([[1, 2, 3, 4]])
_, calls1, acc1, total1 = speculative_decode(draft_model, target_model, prompt1, K=4, max_tokens=20)
n_gen1 = _.shape[1] - prompt1.shape[1]
print(f'  Generated {n_gen1} tokens, Target called {calls1} times, acceptance rate {acc1/total1*100:.0f}%')
print(f'  Effective speedup: {n_gen1/calls1:.1f}x')

# Scenario 2: Random sequence (low acceptance rate simulation)
print()
print('Scenario 2: If Draft and Target differ greatly (simulating low acceptance rate)')
print('  At 30% acceptance rate, K=4 yields ~1.0x speedup (almost no benefit)')
print('  At 90% acceptance rate, K=4 yields ~3.5x speedup')
print()
print('Conclusion: Speculative decoding performance depends heavily on Draft model accuracy.')
print('Choosing the right Draft model is the key factor in making speculative decoding effective.')

## Summary

- Autoregressive generation is serial: each step depends on the previous one
- Speculative decoding uses a small model to guess K tokens, verified by the large model in one pass
- Acceptance/rejection uses the probability ratio min(1, p_target/p_draft), preserving the output distribution
- Multiple experiments confirmed the theoretical acceptance rate
- Speedup depends on Draft accuracy: K=4 + 90% acceptance rate ~ 3.5x
- Variants include: Medusa, Eagle, Self-Speculative, Lookahead
- Best suited for formulaic content; poorly suited for creative writing
- Does not reduce total computation, but reduces waiting time

Next up is Part 4: Frontiers -- Long Context, Chain-of-Thought, and Vision-Language Models.

## Exercises

**Exercise 1: Acceptance Probability Calculation**

The Draft model assigns token A a probability of 0.5, and the Target model assigns it 0.8. What is the acceptance probability?

Hint: min(1, p_target / p_draft)

In [ ]:
p_draft = 0.5
p_target = 0.8
accept_prob = min(1.0, p_target / p_draft)
print(f'Acceptance probability: {accept_prob:.2f}')
assert abs(accept_prob - 1.0) < 0.01
print('Exercise 1 passed: p_target > p_draft, so always accept.')

**Exercise 2: Theoretical Speedup**

K=5, and the acceptance rate per token is 0.8. How many tokens do we expect per Target forward pass?

Hint: Geometric distribution, expected = (1 - 0.8^5) / (1 - 0.8)

In [ ]:
K = 5
alpha = 0.8
expected = (1 - alpha**K) / (1 - alpha)
print(f'Expected tokens per pass: {expected:.2f}')
assert abs(expected - 4.096) < 0.1
print('Exercise 2 passed: K=5, alpha=0.8 yields ~4.1x speedup.')

## References

- Leviathan et al., [Fast Inference from Transformers via Speculative Decoding](https://arxiv.org/abs/2302.01318), 2023
- Chen et al., [Medusa: Simple LLM Inference Acceleration Framework](https://arxiv.org/abs/2401.10774), 2024
- Li et al., [Eagle: Speculative Sampling Requires Rethinking Feature Uncertainty](https://arxiv.org/abs/2401.15077), 2024
- [vLLM Speculative Decoding](https://docs.vllm.ai/en/latest/usage/speculative_decoding.html)
- Harvard NLP, [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/)